# 02 · Train Meridian MMM
Fit a Bayesian Marketing Mix Model using Google Meridian.

In [ ]:
!pip install git+https://github.com/google/meridian.git altair -q

In [ ]:
import numpy as np, pandas as pd
import tensorflow as tf, tensorflow_probability as tfp, arviz as az, IPython
from meridian import constants
from meridian.data import load
from meridian.model import model, spec, prior_distribution
from meridian.analysis import optimizer, visualizer, summarizer
from psutil import virtual_memory

ram_gb = virtual_memory().total / 1e9
print(f'RAM: {ram_gb:.1f} GB')
print('GPUs:', len(tf.config.experimental.list_physical_devices('GPU')))

## Load & Prepare Data

In [ ]:
import sys
sys.path.insert(0, '/content')
from src.data_loader import load_data, detect_columns, build_channel_mappings, build_meridian_dataset

data = load_data('data/synthetic_mmm_data.csv')
media, impressions, output = detect_columns(data)
cost_mapping, impressions_mapping = build_channel_mappings(media, impressions)
data_meridian = build_meridian_dataset(data, media, impressions, output, cost_mapping, impressions_mapping)
data_meridian.as_dataset()

## Define Priors

In [ ]:
def estimate_lognormal_dist(mean, std):
    mu_log = np.log(mean) - 0.5 * np.log((std / mean) ** 2 + 1)
    std_log = np.sqrt(np.log((std / mean) ** 2 + 1))
    return mu_log, std_log

roi_mu, roi_sigma = 2.0, 1.5
roi_mu_log, roi_sigma_log = estimate_lognormal_dist(roi_mu, roi_sigma)
print(f'LogNormal params: mu={roi_mu_log:.3f}, sigma={roi_sigma_log:.3f}')

## Build & Train Model

In [ ]:
prior = prior_distribution.PriorDistribution(
    roi_m=tfp.distributions.LogNormal(
        loc=np.float32(roi_mu_log),
        scale=np.float32(roi_sigma_log),
        name=constants.ROI_M)
)
model_spec = spec.ModelSpec(prior=prior, max_lag=7)
mmm = model.Meridian(input_data=data_meridian, model_spec=model_spec)

In [ ]:
mmm.sample_prior(100)
mmm.sample_posterior(n_chains=5, n_adapt=1000, n_burnin=1000, n_keep=2000)

diagnostics = visualizer.ModelDiagnostics(mmm)
diagnostics.predictive_accuracy_table()

## Save Model Artefacts to Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')